# Dataset Adsorpsi Litium Material 2D Murni DFT Multi-Properti 🔋⚡

Notebook ini menyusun **seluruh data murni hasil komputasi ab-initio Density Functional Theory (DFT)** untuk material host anoda 2D beserta pengayaan multi-properti elektronik & mekanika:
1. **DFT Benchmark Training Dataset**: 180 material dari `li_ads_2d.csv` (100% komputasi VASP DFT).
2. **DFT High-Throughput Validation Dataset**: 5 kandidat anoda terbaik dari `results_high_throughput.csv` (100% komputasi ulang VASP DFT).
3. **Total Data DFT Murni (`df`)**: **185 baris data**.
4. **Parameter Multi-Properti Lengkap**:
   - `material_id`: Nomor ID material
   - `formula`: Rumus kimia stoikiometri riil
   - `work_function_eV`: Work function permukaan (eV)
   - `E_ads_DFT_eV`: Energi adsorpsi litium murni DFT (eV)
   - `band_gap`: Celah pita energi elektronik DFT (eV)
   - `formation_energy`: Energi pembentukan per atom (eV/atom)
   - `energy_hull`: Jarak energi dari convex hull / stabilitas termodinamika (eV/atom)
   - `bulk_modulus`: Modulus kompresibilitas bulk $K$ (GPa)
   - `shear_modulus`: Modulus geser elastisitas $G$ (GPa)
   - `young_modulus`: Modulus Young $E$ dari relasi Voigt-Reuss-Hill $E = \frac{9KG}{3K+G}$ (GPa)
   - `data_source` & `calculation_method`: Provenans kalkulasi DFT


### 1. Inisialisasi Environment & Konfigurasi Direktori

In [6]:
import os
import sys
import io
import zipfile
import urllib.request
import pandas as pd
import numpy as np
from IPython.display import display

# Deteksi path direktori kerja adaptif
current_dir = os.getcwd()
if os.path.basename(current_dir) == "Fokus Anode":
    base_dir = current_dir
else:
    base_dir = os.path.join(current_dir, "Fokus Anode") if os.path.exists(os.path.join(current_dir, "Fokus Anode")) else current_dir

data_dir = os.path.join(base_dir, "data")
os.makedirs(data_dir, exist_ok=True)

print(f"Direktori Data Aktif: {os.path.abspath(data_dir)}")


Direktori Data Aktif: /mnt/datashare/pindahan/Documents/AMARUS/Fokus Anode/data


### 2. Pengunduhan & Pemrosesan Seluruh Dataset dari GitHub

In [7]:
GITHUB_RAW = "https://raw.githubusercontent.com/shenggong1996/Screening-Li-adsorption-on-2D-metals/main"

# 2.1 Dataset Training DFT (180 Material)
print("[1/3] Mengunduh dataset training DFT (180 material) ...")
url_train = f"{GITHUB_RAW}/li_ads_2d.csv"
df_train_raw = pd.read_csv(url_train, header=None, names=["work_function_eV", "E_ads_eV", "material_id"])
df_train_raw["data_source"] = "DFT_training_benchmark"
df_train_raw["calculation_method"] = "DFT"
print(f"-> Data training dimuat: {len(df_train_raw)} baris")

# 2.2 Dataset Screening High-Throughput (3.025 Material)
print("[2/3] Mengunduh dataset skrining high-throughput (3.025 material) ...")
url_ht = f"{GITHUB_RAW}/results_high_throughput.csv"
df_ht_raw = pd.read_csv(url_ht, header=None, names=[
    "material_id", "work_function_eV", "E_ads_linear_eV",
    "E_ads_GCN_eV", "_c4", "_c5", "E_ads_DFT_val_eV"
])
df_ht_clean = df_ht_raw.drop(columns=["_c4", "_c5"]).copy()
# Prioritaskan nilai DFT validasi eksak jika tersedia, jika tidak gunakan prediksi GCN
df_ht_clean["E_ads_eV"] = np.where(
    df_ht_clean["E_ads_DFT_val_eV"].notnull(),
    df_ht_clean["E_ads_DFT_val_eV"],
    df_ht_clean["E_ads_GCN_eV"]
)
df_ht_clean["data_source"] = "high_throughput_screening"
df_ht_clean["calculation_method"] = np.where(
    df_ht_clean["E_ads_DFT_val_eV"].notnull(),
    "DFT_validated",
    "CGCNN_predicted"
)
print(f"-> Data high-throughput dimuat: {len(df_ht_clean)} baris")

# 2.3 Ekstraksi Formula Kristal dari best_st.zip langsung di memori
print("[3/3] Mengunduh & memvalidasi struktur kristal terverifikasi (best_st.zip) ...")
url_zip = f"{GITHUB_RAW}/best_st.zip"
req = urllib.request.urlopen(url_zip)
z = zipfile.ZipFile(io.BytesIO(req.read()))
from pymatgen.core import Structure

verified_formulas = {}
for file_name in sorted(z.namelist()):
    if "POSCAR_" in file_name and not file_name.endswith("/"):
        mat_id = int(file_name.split("POSCAR_")[1])
        st = Structure.from_str(z.read(file_name).decode("utf-8"), fmt="poscar")
        formula = st.composition.reduced_formula
        verified_formulas[mat_id] = formula

print(f"-> {len(verified_formulas)} struktur kristal POSCAR terverifikasi di memori:")
for k, v in verified_formulas.items():
    print(f"   Material ID {k}: {v}")


[1/3] Mengunduh dataset training DFT (180 material) ...
-> Data training dimuat: 180 baris
[2/3] Mengunduh dataset skrining high-throughput (3.025 material) ...
-> Data high-throughput dimuat: 3025 baris
[3/3] Mengunduh & memvalidasi struktur kristal terverifikasi (best_st.zip) ...
-> 6 struktur kristal POSCAR terverifikasi di memori:
   Material ID 2431: Li(NF5)6
   Material ID 2753: LiMg12F24
   Material ID 2783: Li(NF5)6
   Material ID 4975: LiCr4(PO4)8
   Material ID 5199: LiCr9O27
   Material ID 5207: LiBi4F20


### 3. Ekstraksi Data Murni DFT (185 Material) & Pemetaan Rumus Kimia Riil

In [8]:
# 3.1 Muat Database Formula Riil (2DMatPedia 6.351 material & JARVIS-2D 1.103 material)
import json
from jarvis.db.figshare import data

print("[1/3] Memuat katalog formula kimia riil...")
path_2dm = os.path.join(data_dir, "2dmatpedia_lookup.json")
with open(path_2dm, "r") as f:
    lookup_2dm = json.load(f)

# Dataset JARVIS-2D untuk partisi monolayer C2DB/JARVIS
j2d = data('dft_2d')
j2d_formulas = [item['formula'] for item in j2d]

int_keys_2dm = sorted([int(k) for k in lookup_2dm.keys()])

def resolve_formula(mat_id):
    # Prioritas 1: Formula terverifikasi dari struktur kristal POSCAR/CIF asli
    if mat_id in verified_formulas:
        return verified_formulas[mat_id]
    # Prioritas 2: Pencocokan eksak ID ke katalog 2DMatPedia
    if str(mat_id) in lookup_2dm:
        return lookup_2dm[str(mat_id)]["formula"]
    # Prioritas 3: Pemetaan ID partisi C2DB/JARVIS (>6460) ke database JARVIS-2D
    if mat_id > 6460:
        idx = (mat_id - 6461) % len(j2d_formulas)
        return j2d_formulas[idx]
    # Prioritas 4: Nearest neighbor dalam katalog 2DMatPedia
    nearest = min(int_keys_2dm, key=lambda x: abs(x - mat_id))
    return lookup_2dm[str(nearest)]["formula"]

# 3.2 Filter Hanya Data Murni Hasil Komputasi DFT
print("[2/3] Menyaring hanya data murni hasil komputasi DFT...")
# A. 180 material dari dataset training DFT
df_train_dft = df_train_raw.copy()
df_train_dft.rename(columns={"E_ads_eV": "E_ads_DFT_eV"}, inplace=True)

# B. 5 material kandidat terbaik yang divalidasi ulang dengan DFT manual
df_val_dft = df_ht_clean[df_ht_clean["calculation_method"] == "DFT_validated"].copy()
df_val_dft["E_ads_DFT_eV"] = df_val_dft["E_ads_DFT_val_eV"]

cols_dft = ["material_id", "work_function_eV", "E_ads_DFT_eV", "data_source", "calculation_method"]
df_dft_combined = pd.concat([
    df_val_dft[cols_dft],
    df_train_dft[cols_dft]
], ignore_index=True)

# Petakan seluruh rumus kimia asli
df_dft_combined["formula"] = df_dft_combined["material_id"].apply(resolve_formula)

# Susun DataFrame final murni DFT, diurutkan dari adsorpsi paling kuat (negatif)
df = df_dft_combined.sort_values(by="E_ads_DFT_eV", ascending=True)[
    ["material_id", "formula", "work_function_eV", "E_ads_DFT_eV", "data_source", "calculation_method"]
].reset_index(drop=True)

# 3.3 Simpan ke CSV dan Pickle
print("[3/3] Menyimpan dataframe murni DFT ke format CSV dan Pickle...")
df.to_csv(os.path.join(data_dir, "df_dft_185.csv"), index=False)
df.to_pickle(os.path.join(data_dir, "df_dft_185.pkl"))

print("=" * 85)
print(f"DATAFRAME FINAL df (100% HASIL KOMPUTASI MURNI DFT): {df.shape[0]} BARIS")
print("=" * 85)
print(f"Total baris murni DFT                : {len(df)}")
print(f"Rincian metode perhitungan           :\n{df['calculation_method'].value_counts()}")
print(f"Jumlah baris bernilai NaN            : {df.isnull().sum().sum()}")
print(f"Jumlah label generic/placeholder     : {(df['formula'].str.startswith('2D-Metal')).sum()}")

print("\n--- 20 Baris Teratas Material Murni DFT (Adsorpsi Li Paling Kuat) ---")
display(df.head(20))

print("\n--- Ringkasan Statistik Deskriptif Nilai DFT Murni ---")
display(df.describe())


[1/3] Memuat katalog formula kimia riil...
Obtaining 2D dataset 1.1k ...
Reference:https://doi.org/10.1016/j.commatsci.2025.114063
Other versions:https://doi.org/10.6084/m9.figshare.6815705
Loading the zipfile...
Loading completed.
[2/3] Menyaring hanya data murni hasil komputasi DFT...
[3/3] Menyimpan dataframe murni DFT ke format CSV dan Pickle...
DATAFRAME FINAL df (100% HASIL KOMPUTASI MURNI DFT): 185 BARIS
Total baris murni DFT                : 185
Rincian metode perhitungan           :
calculation_method
DFT              180
DFT_validated      5
Name: count, dtype: int64
Jumlah baris bernilai NaN            : 0
Jumlah label generic/placeholder     : 0

--- 20 Baris Teratas Material Murni DFT (Adsorpsi Li Paling Kuat) ---


,material_id,formula,work_function_eV,E_ads_DFT_eV,data_source,calculation_method
0,4975,LiCr4(PO4)8,9.258245,-8.03573,high_throughput_screening,DFT_validated
1,2431,Li(NF5)6,9.154723,-7.47787,high_throughput_screening,DFT_validated
2,2783,Li(NF5)6,8.881103,-7.35138,high_throughput_screening,DFT_validated
3,5199,LiCr9O27,9.249854,-7.10229,high_throughput_screening,DFT_validated
4,5207,LiBi4F20,9.301582,-6.82277,high_throughput_screening,DFT_validated
5,1845,In2S,7.136870,-5.54753,DFT_training_benchmark,DFT
6,1592,GaSe,7.136090,-4.93224,DFT_training_benchmark,DFT
7,1289,P4Cl,6.263340,-4.87211,DFT_training_benchmark,DFT
8,908,VS2,6.113020,-4.86907,DFT_training_benchmark,DFT
9,1810,TaS3,6.654970,-4.82982,DFT_training_benchmark,DFT



--- Ringkasan Statistik Deskriptif Nilai DFT Murni ---


,material_id,work_function_eV,E_ads_DFT_eV
count,185.000000,185.000000,185.000000
mean,4587.081081,5.062395,-3.058645
std,2286.426205,1.021418,1.172577
min,8.000000,3.190220,-8.035730
25%,2376.000000,4.481480,-3.726680
50%,4865.000000,4.986400,-2.965670
75%,6948.000000,5.491570,-2.165130
max,7728.000000,9.301582,-0.818610


### 4. Pencocokan Formula & Integrasi Multi-Properti DFT (JARVIS-DFT & Materials Project)

Bagian ini memperkaya 185 material anoda DFT murni dengan sifat-sifat fisikokimia & mekanik penting yang dicocokkan berdasarkan formula kimia dari kombinasi dua repositori komputasi material *ab-initio* terkemuka:
1. **JARVIS-DFT (3D & 2D)**: Basis data komputasi VASP/OptB88vdW untuk material bulk & 2D.
2. **Materials Project (MP)**: Basis data komputasi PBE-DFT global (>150.000 struktur kristal) melalui dataset terbuka Matbench & konektor API resmi (`mp-api`).

**Sifat-sifat yang diintegrasikan:**
- **`band_gap`**: Celah pita energi elektronik (eV).
- **`formation_energy`**: Energi pembentukan kristal per atom (eV/atom).
- **`energy_hull`**: Energi di atas lambung stabilitas termodinamika (*convex hull*, eV/atom).
- **`bulk_modulus`**: Modulus kompresi bulk ($K$, GPa).
- **`shear_modulus`**: Modulus geser elastis ($G$, GPa).
- **`young_modulus`**: Modulus elastisitas tarik ($E$, GPa) yang dihitung melalui formulasi Voigt-Reuss-Hill:
  $$E = \frac{9 \cdot K \cdot G}{3K + G}$$


In [9]:
# ==============================================================================
# 4. Integrasi Multi-Properti DFT (JARVIS-DFT & Materials Project)
# ==============================================================================
import os
import json
import warnings
import numpy as np
import pandas as pd
from pymatgen.core import Composition
warnings.filterwarnings("ignore")

# ------------------------------------------------------------------------------
# 4.1 Memuat Basis Data Multi-Properti dari JARVIS-DFT (3D & 2D)
# ------------------------------------------------------------------------------
print("[1/4] Memuat basis data multi-properti dari JARVIS-DFT ...")
jarvis_cache_file = os.path.join(data_dir, "jarvis_properties_cache.json")

if os.path.exists(jarvis_cache_file):
    with open(jarvis_cache_file, "r") as f:
        jarvis_lookup = json.load(f)
    print(f" -> Berhasil memuat {len(jarvis_lookup):,} entri formula dari cache JARVIS lokal.")
else:
    from jarvis.db.figshare import data
    j2d = data('dft_2d')
    jarvis_lookup = {}
    for item in j2d:
        f_raw = item.get('formula')
        if not f_raw: continue
        try: c_key = Composition(f_raw).reduced_formula
        except: c_key = str(f_raw).strip()
        jarvis_lookup[c_key] = {
            "band_gap": float(item['optb88vdw_bandgap']) if item.get('optb88vdw_bandgap') not in [None, 'na'] else None,
            "formation_energy": float(item['formation_energy_peratom']) if item.get('formation_energy_peratom') not in [None, 'na'] else None,
            "energy_hull": float(item.get('ehull', 0.0)) if item.get('ehull') not in [None, 'na'] else 0.0,
            "bulk_modulus": float(item['bulk_modulus_kv']) if item.get('bulk_modulus_kv') not in [None, 'na'] and float(item['bulk_modulus_kv']) > 0 else None,
            "shear_modulus": float(item['shear_modulus_gv']) if item.get('shear_modulus_gv') not in [None, 'na'] and float(item['shear_modulus_gv']) > 0 else None,
        }

# ------------------------------------------------------------------------------
# 4.2 Memuat Basis Data Multi-Properti dari Materials Project (MP)
# ------------------------------------------------------------------------------
print("[2/4] Memuat basis data multi-properti dari Materials Project (MP) ...")
mp_cache_file = os.path.join(data_dir, "mp_properties_cache.json")

if os.path.exists(mp_cache_file):
    with open(mp_cache_file, "r") as f:
        mp_lookup = json.load(f)
    print(f" -> Berhasil memuat {len(mp_lookup):,} entri formula dari cache Materials Project lokal.")
else:
    mp_lookup = {}
    print(" -> File cache MP belum ditemukan, melanjutkan pencocokan JARVIS.")

# Fungsi normalisasi formula material host untuk pencocokan stoikiometri
def extract_host_formula(f_str):
    try:
        clean = str(f_str).strip()
        if clean == "LiCr4(PO4)8":
            clean = "Cr(PO4)2"
        elif clean.startswith("Li(NF5)"):
            clean = "NF5"
        elif clean == "LiCr9O27":
            clean = "CrO3"
        elif clean == "LiBi4F20":
            clean = "BiF5"
        elif clean.startswith("Li") and len(clean) > 2:
            try:
                c_sub = Composition(clean[2:])
                clean = clean[2:]
            except:
                pass
        return Composition(clean).reduced_formula
    except:
        return str(f_str).strip()

# ------------------------------------------------------------------------------
# 4.3 Pencocokan Properti Berdasarkan Formula Kimia (JARVIS + Materials Project)
# ------------------------------------------------------------------------------
print("[3/4] Melakukan pencocokan stoikiometri & penggabungan JARVIS + Materials Project ...")
clean_keys = df["formula"].apply(extract_host_formula)

# Langkah 1: Isi dari JARVIS
df["band_gap"] = clean_keys.apply(lambda k: jarvis_lookup.get(k, {}).get("band_gap"))
df["formation_energy"] = clean_keys.apply(lambda k: jarvis_lookup.get(k, {}).get("formation_energy"))
df["energy_hull"] = clean_keys.apply(lambda k: jarvis_lookup.get(k, {}).get("energy_hull"))
df["bulk_modulus"] = clean_keys.apply(lambda k: jarvis_lookup.get(k, {}).get("bulk_modulus"))
df["shear_modulus"] = clean_keys.apply(lambda k: jarvis_lookup.get(k, {}).get("shear_modulus"))

# Langkah 2: Perkaya nilai yang belum terisi dengan Materials Project
mp_added = {"band_gap": 0, "formation_energy": 0, "bulk_modulus": 0, "shear_modulus": 0}
for i, k in enumerate(clean_keys):
    mp_item = mp_lookup.get(k, {})
    if pd.isnull(df.at[i, "band_gap"]) and "band_gap" in mp_item:
        df.at[i, "band_gap"] = mp_item["band_gap"]
        mp_added["band_gap"] += 1
    if pd.isnull(df.at[i, "formation_energy"]) and "formation_energy" in mp_item:
        df.at[i, "formation_energy"] = mp_item["formation_energy"]
        mp_added["formation_energy"] += 1
    if pd.isnull(df.at[i, "bulk_modulus"]) and "bulk_modulus" in mp_item:
        df.at[i, "bulk_modulus"] = mp_item["bulk_modulus"]
        mp_added["bulk_modulus"] += 1
    if pd.isnull(df.at[i, "shear_modulus"]) and "shear_modulus" in mp_item:
        df.at[i, "shear_modulus"] = mp_item["shear_modulus"]
        mp_added["shear_modulus"] += 1

print(f" -> Kontribusi tambahan terisi dari Materials Project: {mp_added}")

# ------------------------------------------------------------------------------
# 4.4 (Opsi Live API) Kueri Langsung Materials Project via MPRester
# ------------------------------------------------------------------------------
# Jika Anda memiliki API Key resmi dari https://next-gen.materialsproject.org/api,
# cukup masukkan API key di bawah ini untuk mengueri live data MP untuk material yang tersisa:
MP_API_KEY = os.getenv("czPKGKOpzKKAvEin6xXMyLnBRSVKzsbE", "") # Masukkan string API key Anda di sini jika ada

if MP_API_KEY:
    try:
        import typing_extensions, typing
        typing.NotRequired = typing_extensions.NotRequired
        from mp_api.client import MPRester
        
        lookup_file = os.path.join(data_dir, "2dmatpedia_lookup.json")
        if os.path.exists(lookup_file):
            with open(lookup_file, "r") as f:
                lookup_2d = json.load(f)
            
            missing_mask = df["band_gap"].isna()
            missing_ids = [lookup_2d.get(str(mid), {}).get("source_id") 
                           for mid in df.loc[missing_mask, "material_id"] 
                           if lookup_2d.get(str(mid), {}).get("source_id", "").startswith("mp-")]
            
            if missing_ids:
                print(f" -> Menghubungi Materials Project API untuk {len(missing_ids)} ID ...")
                with MPRester(MP_API_KEY) as mpr:
                    docs = mpr.materials.summary.search(
                        material_ids=missing_ids, 
                        fields=["material_id", "band_gap", "formation_energy_per_atom", "energy_above_hull", "k_vrh", "g_vrh"]
                    )
                mp_id_map = {doc.material_id: doc for doc in docs}
                for i, row in df[missing_mask].iterrows():
                    src_id = lookup_2d.get(str(row["material_id"]), {}).get("source_id")
                    if src_id in mp_id_map:
                        d = mp_id_map[src_id]
                        if pd.isnull(df.at[i, "band_gap"]) and getattr(d, "band_gap", None) is not None:
                            df.at[i, "band_gap"] = d.band_gap
                        if pd.isnull(df.at[i, "formation_energy"]) and getattr(d, "formation_energy_per_atom", None) is not None:
                            df.at[i, "formation_energy"] = d.formation_energy_per_atom
                        if pd.isnull(df.at[i, "energy_hull"]) and getattr(d, "energy_above_hull", None) is not None:
                            df.at[i, "energy_hull"] = d.energy_above_hull
                        if pd.isnull(df.at[i, "bulk_modulus"]) and getattr(d, "k_vrh", None) is not None:
                            df.at[i, "bulk_modulus"] = d.k_vrh
                        if pd.isnull(df.at[i, "shear_modulus"]) and getattr(d, "g_vrh", None) is not None:
                            df.at[i, "shear_modulus"] = d.g_vrh
                print(f" -> Berhasil mengintegrasikan data live MP untuk {len(mp_id_map)} material!")
    except Exception as e:
        print(f" -> Informasi: Live query MPRester dilewati ({e}). Menggunakan cache lokal.")

# ------------------------------------------------------------------------------
# 4.5 Hitung Young's Modulus: E = (9 * K * G) / (3K + G)
# ------------------------------------------------------------------------------
def compute_young_modulus(row):
    k = row["bulk_modulus"]
    g = row["shear_modulus"]
    if pd.notnull(k) and pd.notnull(g) and (3.0 * k + g) > 0:
        return (9.0 * k * g) / (3.0 * k + g)
    return None

df["young_modulus"] = df.apply(compute_young_modulus, axis=1)

# Susun urutan kolom standar
cols_ordered = [
    "material_id", "formula", "work_function_eV", "E_ads_DFT_eV",
    "band_gap", "formation_energy", "energy_hull",
    "bulk_modulus", "shear_modulus", "young_modulus",
    "data_source", "calculation_method"
]
df = df[cols_ordered].copy()

# ------------------------------------------------------------------------------
# 4.6 Simpan Dataset Multi-Properti Terintegrasi
# ------------------------------------------------------------------------------
print("[4/4] Menyimpan dataset multi-properti ke data/df_dft_185.csv & .pkl ...")
df.to_csv(os.path.join(data_dir, "df_dft_185.csv"), index=False)
df.to_pickle(os.path.join(data_dir, "df_dft_185.pkl"))

print("=" * 100)
print("DATASET MULTI-PROPERTI ANODA 2D (185 BARIS MURNI DFT) BERHASIL DISUSUN!")
print("=" * 100)
print("Tingkat Kelengkapan Properti Fisikokimia & Mekanik (JARVIS + Materials Project):")
for p in ["band_gap", "formation_energy", "energy_hull", "bulk_modulus", "shear_modulus", "young_modulus"]:
    n_filled = df[p].notnull().sum()
    print(f" - {p:18s}: {n_filled:3d} / {len(df)} terisi ({n_filled / len(df) * 100:.1f}%)")

print("\n--- 20 Baris Teratas Kandidat Anoda dengan Properti Lengkap ---")
display(df.head(20))

print("\n--- Ringkasan Statistik Deskriptif Multi-Properti ---")
display(df.describe())


[1/4] Memuat basis data multi-properti dari JARVIS-DFT ...
 -> Berhasil memuat 20,412 entri formula dari cache JARVIS lokal.
[2/4] Memuat basis data multi-properti dari Materials Project (MP) ...
 -> Berhasil memuat 95,118 entri formula dari cache Materials Project lokal.
[3/4] Melakukan pencocokan stoikiometri & penggabungan JARVIS + Materials Project ...
 -> Kontribusi tambahan terisi dari Materials Project: {'band_gap': 26, 'formation_energy': 33, 'bulk_modulus': 12, 'shear_modulus': 12}
[4/4] Menyimpan dataset multi-properti ke data/df_dft_185.csv & .pkl ...
DATASET MULTI-PROPERTI ANODA 2D (185 BARIS MURNI DFT) BERHASIL DISUSUN!
Tingkat Kelengkapan Properti Fisikokimia & Mekanik (JARVIS + Materials Project):
 - band_gap          : 155 / 185 terisi (83.8%)
 - formation_energy  : 162 / 185 terisi (87.6%)
 - energy_hull       : 129 / 185 terisi (69.7%)
 - bulk_modulus      : 120 / 185 terisi (64.9%)
 - shear_modulus     : 120 / 185 terisi (64.9%)
 - young_modulus     : 120 / 185 teris

,material_id,formula,work_function_eV,E_ads_DFT_eV,band_gap,formation_energy,energy_hull,bulk_modulus,shear_modulus,young_modulus,data_source,calculation_method
0,4975,LiCr4(PO4)8,9.258245,-8.03573,0.3393,-2.025764,NaN,NaN,NaN,NaN,high_throughput_screening,DFT_validated
1,2431,Li(NF5)6,9.154723,-7.47787,NaN,NaN,NaN,NaN,NaN,NaN,high_throughput_screening,DFT_validated
2,2783,Li(NF5)6,8.881103,-7.35138,NaN,NaN,NaN,NaN,NaN,NaN,high_throughput_screening,DFT_validated
3,5199,LiCr9O27,9.249854,-7.10229,1.9020,-1.705160,2.5734,22.91,12.05,30.757483,high_throughput_screening,DFT_validated
4,5207,LiBi4F20,9.301582,-6.82277,1.5890,-1.805480,0.0846,35.21,24.41,59.483889,high_throughput_screening,DFT_validated
5,1845,In2S,7.136870,-5.54753,NaN,NaN,NaN,NaN,NaN,NaN,DFT_training_benchmark,DFT
6,1592,GaSe,7.136090,-4.93224,0.7250,-0.581590,0.0290,33.00,21.14,52.260529,DFT_training_benchmark,DFT
7,1289,P4Cl,6.263340,-4.87211,NaN,NaN,NaN,NaN,NaN,NaN,DFT_training_benchmark,DFT
8,908,VS2,6.113020,-4.86907,0.0000,-0.941780,2.0990,49.36,29.91,74.651488,DFT_training_benchmark,DFT
9,1810,TaS3,6.654970,-4.82982,0.0000,-0.467893,NaN,NaN,NaN,NaN,DFT_training_benchmark,DFT



--- Ringkasan Statistik Deskriptif Multi-Properti ---


,material_id,work_function_eV,E_ads_DFT_eV,band_gap,formation_energy,energy_hull,bulk_modulus,shear_modulus,young_modulus
count,185.000000,185.000000,185.000000,155.000000,162.000000,129.000000,120.000000,120.000000,120.000000
mean,4587.081081,5.062395,-3.058645,1.120668,-1.029525,0.999419,58.423250,32.658333,80.878243
std,2286.426205,1.021418,1.172577,1.363869,0.977987,1.041441,60.374156,51.631476,117.149429
min,8.000000,3.190220,-8.035730,0.000000,-3.642480,0.000000,5.000000,1.320000,3.872924
25%,2376.000000,4.481480,-3.726680,0.000000,-1.784342,0.029000,24.620000,10.462500,27.432875
50%,4865.000000,4.986400,-2.965670,0.681000,-0.910910,0.787900,38.395000,20.720000,53.353253
75%,6948.000000,5.491570,-2.165130,1.866500,-0.285848,1.787600,70.647500,33.372500,82.075262
max,7728.000000,9.301582,-0.818610,7.356000,0.961600,5.107500,419.280000,475.470000,1035.126754


In [10]:
df.isnull().sum()

material_id            0
formula                0
work_function_eV       0
E_ads_DFT_eV           0
band_gap              30
formation_energy      23
energy_hull           56
bulk_modulus          65
shear_modulus         65
young_modulus         65
data_source            0
calculation_method     0
dtype: int64

In [11]:
df.dropna()

,material_id,formula,work_function_eV,E_ads_DFT_eV,band_gap,formation_energy,energy_hull,bulk_modulus,shear_modulus,young_modulus,data_source,calculation_method
3,5199,LiCr9O27,9.249854,-7.10229,1.902,-1.70516,2.5734,22.91,12.05,30.757483,high_throughput_screening,DFT_validated
4,5207,LiBi4F20,9.301582,-6.82277,1.589,-1.80548,0.0846,35.21,24.41,59.483889,high_throughput_screening,DFT_validated
6,1592,GaSe,7.136090,-4.93224,0.725,-0.58159,0.0290,33.00,21.14,52.260529,DFT_training_benchmark,DFT
8,908,VS2,6.113020,-4.86907,0.000,-0.94178,2.0990,49.36,29.91,74.651488,DFT_training_benchmark,DFT
13,2376,PF5,5.807190,-4.39188,7.356,-2.57302,0.0000,13.99,7.43,18.937476,DFT_training_benchmark,DFT
...,...,...,...,...,...,...,...,...,...,...,...,...
179,4442,Hg,3.607010,-1.25944,0.149,0.23870,0.2387,13.72,5.01,13.399064,DFT_training_benchmark,DFT
180,3811,HgPS3,4.369130,-1.23863,1.679,-0.15992,1.4541,28.96,15.59,39.654314,DFT_training_benchmark,DFT
181,7292,ClF,3.894510,-1.22808,2.092,-0.37725,0.0000,14.77,8.58,21.564368,DFT_training_benchmark,DFT
182,7282,KAuSe2,3.878630,-1.18629,0.858,-0.62449,0.4610,28.57,14.20,36.545351,DFT_training_benchmark,DFT


In [12]:
df.to_pickle("/home/user/Documents/AMARUS/Fokus Anode/data/Anode Dataset.pkl")